# multiply-back — worked example 1: multiply_back with column vector times row vector broadcasting

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `multiply-back`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The backward pass for `out = x * y` computes `dL/dx = grad_out * y` and `dL/dy = grad_out * x`, then collapses any dimensions that were broadcast during the forward pass. `unbroadcast(grad, parent)` achieves this by summing over leading singleton-expanded axes and any prepended batch axes, returning a tensor with the same shape as the original (pre-broadcast) parent.

## Worked solution

**Step 1 — identify the broadcast pattern.** If `x` has shape `(N, 1)` and `y` has shape `(1, M)`, broadcasting produces `out` of shape `(N, M)`. The backward for `x` must collapse the `M` axis (dim=1 was broadcast), and the backward for `y` must collapse the `N` axis (dim=0 was broadcast).

**Step 2 — compute the raw product.** For `multiply_back0` (grad for `x`): `raw = grad_out * y`. For `multiply_back1` (grad for `y`): `raw = grad_out * x`. These products still have shape `(N, M)` due to broadcasting.

**Step 3 — call `unbroadcast(raw, parent)`.** This sums `raw` over whatever axes were introduced by broadcasting, returning a tensor of the same shape as `x` or `y` respectively.

**Step 4 — verify against autograd.** The same backward pass can be computed by PyTorch's autograd. Our hand-written result should match to within floating-point tolerance.

In [ ]:
import torch
from torch import Tensor

torch.manual_seed(0)

def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    """Collapse gradient back to original's shape by summing broadcast axes."""
    # Sum over leading dims that were prepended
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    # Sum over dims that were size-1 in original but expanded in grad
    for i, (gs, os) in enumerate(zip(grad.shape, original.shape)):
        if os == 1 and gs != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def multiply_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """dL/dx = unbroadcast(grad_out * y, x)"""
    if not isinstance(y, Tensor):
        y = torch.tensor(y, dtype=grad_out.dtype)
    return unbroadcast(grad_out * y, x)

def multiply_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """dL/dy = unbroadcast(grad_out * x, y)"""
    if not isinstance(x, Tensor):
        x = torch.tensor(x, dtype=grad_out.dtype)
    return unbroadcast(grad_out * x, y)

# Exercise: column vector (4,1) times row vector (1,5)
torch.manual_seed(2)
x = torch.randn(4, 1, requires_grad=True)
y = torch.randn(1, 5, requires_grad=True)
out = x * y  # (4, 5)
loss = out.sum()
loss.backward()

# Our hand-written backward
grad_out = torch.ones(4, 5)
our_gx = multiply_back0(grad_out, out, x.detach(), y.detach())
our_gy = multiply_back1(grad_out, out, x.detach(), y.detach())

print(f"x.grad shape:   {x.grad.shape}, our shape:   {our_gx.shape}")
print(f"y.grad shape:   {y.grad.shape}, our shape:   {our_gy.shape}")
assert torch.allclose(our_gx, x.grad, atol=1e-6), "gx mismatch"
assert torch.allclose(our_gy, y.grad, atol=1e-6), "gy mismatch"
print("multiply_back0 and multiply_back1 match autograd!")